# The object graph: nodes and edges

Shows one run's graph: every node (one object, merged from SAM tracks by `graph/graph.py`) as its 3D box, and the
edges between them (`graph/edges.py`):

- **on**: one object rests on or in another, such as a cushion on a sofa. Drawn as an arrow from the object on top.
- **near**: the two boxes are at most 25 cm apart. Drawn as a dashed line.

Walls, floors, ceilings and nodes with a box side over 3 m get no edges and aren't drawn.

Run `run.py` on the run first, since this reads its cached stages:

```bash
uv run --no-sync python run.py --source replica/room_0/00
```

In [ ]:
import warnings

warnings.filterwarnings("ignore", category=FutureWarning)  # torch.jit, via transformers

from collections import Counter

from graph import edges as graph_edges
from graph import plot_edges
from utils import footage

SOURCE = "replica/room_0/00"  # or phone/<name> for a phone run
STRIDE = 20
MASK_SOURCE = "sam3"  # or "gt" for the GT masks
POSE_SOURCE = None  # None: Replica's poses on Replica, VGGT-Omega's on phone video

run = footage.make_run(SOURCE, STRIDE, MASK_SOURCE, POSE_SOURCE, "")
nodes, edges = plot_edges.load_graph(run)
counts = Counter(e.relation for e in edges)
print(f"{run.source}: {len(nodes)} nodes, {len(edges)} edges ({counts['on']} on, {counts['near']} near)")

## The graph from above

Each node is its 3D box seen from above, over the room's point cloud, in the same colour as in the labelled video.
Set `near=False` to see only the "on" edges.

In [ ]:
plot_edges.plot(run, near=True);

## The "on" edges

What rests on what. These are the edges that tell similar objects apart: "the cushion on the armchair" rather than
"a cushion".

In [ ]:
for e in sorted(edges, key=lambda e: (e.b, e.a)):
    if e.relation == "on":
        print(f"{nodes[e.a].label:>14} {e.a:<3} on  {nodes[e.b].label} {e.b}")

## One node and its neighbours

Pick a node ID from the picture above to see its edges and highlight it with its neighbours.

In [ ]:
NODE = 37

n = nodes[NODE]
print(f"node {n.id}: {n.label}, seen in {len(n.frames)} frames, SAM tracks {n.sam_ids}")
links = graph_edges.neighbours(edges, NODE)
for relation, other in links:
    print(f"  {relation:<5} {nodes[other].label} {other}")
plot_edges.plot(run, highlight=[NODE] + [other for _, other in links]);

## Nearest object of a type

What `graph.query --near` does: find the object that best matches `NEAR` by SigLIP, then rank the objects of type
`TYPE` by distance to it. The type is the SAM label when `TYPE` is one of the prompts, otherwise SigLIP's best
matches. The anchor and the nearest answer are highlighted.

In [ ]:
from graph import query
from models import siglip
from utils.schema import Graph, Masks, TextAligned, load_stage

TYPE, NEAR = "cushion", "sofa"

masks = load_stage(run.stage_dir("masks"), Masks)
node_of_mask = load_stage(run.stage_dir("graph"), Graph).node
objects, object_emb = query.object_embeddings(masks, load_stage(run.stage_dir("siglip"), TextAligned), node_of_mask)
sims = siglip.load_encoder().encode_text([TYPE, NEAR]).cpu().numpy() @ object_emb.T
anchor = int(objects[sims[1].argmax()])
rows = query.of_type([nodes[int(o)].label for o in objects], TYPE, sims[0], top=5)
ranked = graph_edges.nearest(nodes, anchor, [int(objects[r]) for r in rows])

print(f'"{NEAR}": node {anchor} ({nodes[anchor].label})')
for node_id, gap, centre in ranked[:5]:
    print(f"  {nodes[node_id].label} {node_id}: {gap:.2f} m apart, {centre:.2f} m between centres")
plot_edges.plot(run, highlight=[anchor, ranked[0][0]] if ranked else [anchor], near=False);

## In 3D

The room's point cloud, the camera path and every object as a labelled box in Rerun. The edges are lines between the
box centres: `world/edges/on` as black arrows and `world/edges/near` in grey. Hide either in the entity tree on the
left, and hover a line for the two objects it joins. Scrub the `frame` timeline to move the camera.

In [ ]:
import rerun as rr

from utils import map_view

rr.init("graph")
rr.notebook_show(height=750)
map_view.log_map(run)

## How well the nearest-object query does

`run.py` scores it against the GT boxes (Replica only, see the README's Evaluation section): for every GT object and
every class with at least two other objects, is the nearest one the map gives the right one?

- **labels**: candidates are the nodes with that SAM label, as above
- **oracle**: candidates are the nodes that stand for an object of that class: only the map's boxes and identities
- **anchor** / **reach**: share of the queries whose anchor is in the map / whose anchor and answer both are

In [ ]:
import json

metrics = json.loads((run.stage_dir("eval") / "metrics.json").read_text())
print(f"{'':<18}{'labels':>8}{'oracle':>8}{'anchor':>8}{'reach':>8}{'queries':>9}")
for name, m in metrics.items():
    if "near_labels" in m:
        print(
            f"{name:<18}{m['near_labels']:>8.2f}{m['near_oracle']:>8.2f}{m['near_anchor']:>8.2f}"
            f"{m['near_reach']:>8.2f}{m['near_queries']:>9}"
        )